# Step 01 — OSM extraction and preparation

Clip the regional OSM extract to the study area and prepare two layers.

| | |
|---|---|
| **Reads** | `data/input/regionalverband_area.gpkg`, `data/input/niedersachsen-*.osm.pbf` |
| **Writes** | `01_all_pois.gpkg` · `01_all_buildings_osm.gpkg` · `01_study_area_clipped.pbf` |
| **Needs** | `osmium` (system binary), `pyrosm` |
| **Runtime** | 3-5 min, two `pyrosm` parses of the clipped PBF |

**`01_all_pois.gpkg`** — the activity layer. Filtered to POIs that describe
something happening inside a building — read from all three ways a mapper says
it: a use tag on a node or building, a named building with only a
`building=<use>` tag, or an area drawn around the site — each carrying `poi_use` (what happens
here) and `poi_role` (how it can be joined to a building). A POI nested inside
another POI's polygon — a shop in a mall, an institute on a campus — also
carries `poi_parent_id`, and a shop in a mall gets the role `unit` and a
`split_area_m2` for sharing the building's volume (section 8).

**`01_all_buildings_osm.gpkg`** — every OSM building footprint, polygons only.
Not filtered by use: an unlabelled `building=yes` is still a real building.

Sections 1–2 clip. Section 3 prepares the buildings, 4–7 the POIs — in that
order because the role classification needs the building geometry. Section 8
resolves which POI sits inside which. Section 9 writes both, so a failure above
cannot leave one output newer than the other.

In [1]:
import os, shutil, sys
from pathlib import Path

# --- locate the pipeline root -------------------------------------------------
# Do NOT use Path('..') here: a notebook's working directory is not necessarily
# its own folder. VS Code starts kernels in ${workspaceFolder} by default, so
# whenever the workspace is opened above this repo, '..' points somewhere else
# entirely and `import config` fails. Find the root by its marker file instead.
def _find_root(start):
    for d in (start, *start.parents):
        if (d / 'config.py').is_file() and (d / 'lib' / 'checks.py').is_file():
            return d
    return None

_nb_dir = Path(globals()['__vsc_ipynb_file__']).parent if '__vsc_ipynb_file__' in globals() else None
ROOT_DIR = _find_root(_nb_dir) if _nb_dir else None
ROOT_DIR = ROOT_DIR or _find_root(Path.cwd())
if ROOT_DIR is None:
    raise RuntimeError(
        'Cannot find the pipeline root (the folder containing config.py). '
        f'Looked upward from notebook dir {_nb_dir} and cwd {Path.cwd()}.'
    )
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

# --- point GDAL/PROJ at this env's data files ---------------------------------
# Without these, pyogrio warns on every read and write and CRS lookups can fail
# outright. Must run before geopandas is imported; only sets what is missing.
_share = Path(sys.prefix) / 'Library' / 'share'
if not _share.is_dir():
    _share = Path(sys.prefix) / 'share'
if (_share / 'gdal').is_dir():
    os.environ.setdefault('GDAL_DATA', str(_share / 'gdal'))
if (_share / 'proj').is_dir():
    os.environ.setdefault('PROJ_LIB', str(_share / 'proj'))

import subprocess, tempfile, json, time

# --- find a working osmium ----------------------------------------------------
# Two separate traps, both silent.
#
# 1. A kernel started WITHOUT `conda activate` - which is how VS Code launches
#    one - has none of this environment's binaries on PATH. A bare
#    subprocess.run(['osmium', ...]) then dies with
#    `FileNotFoundError: [WinError 2] The system cannot find the file
#    specified`, naming neither osmium nor PATH.
#
# 2. `shutil.which('osmium')` returns `osmium.EXE` on Windows, upper-cased from
#    PATHEXT. osmium-tool 1.19 parses its own argv[0], does not recognise the
#    upper-case name, and exits 2 with `Unknown command or option 'osmium.EXE'`.
#    The identical binary spelled `osmium.exe` works.
#
# So candidates are verified by running them, never assumed. Preferring this
# env's copy also pins the version - base holds an older osmium that would
# otherwise win on PATH.
for _bin in (Path(sys.prefix) / 'Library' / 'bin',
             Path(sys.prefix) / 'Scripts',
             Path(sys.prefix) / 'bin'):
    if _bin.is_dir() and str(_bin) not in os.environ.get('PATH', ''):
        os.environ['PATH'] = str(_bin) + os.pathsep + os.environ.get('PATH', '')


def _find_osmium():
    cands = []
    for d in (Path(sys.prefix) / 'Library' / 'bin',
              Path(sys.prefix) / 'Scripts',
              Path(sys.prefix) / 'bin'):
        cands += [d / 'osmium.exe', d / 'osmium']
    found = shutil.which('osmium')
    if found:
        f = Path(found)
        cands += [f.with_suffix(f.suffix.lower()), f]
    for c in cands:
        if not c.is_file():
            continue
        try:
            r = subprocess.run([str(c), '--version'], capture_output=True, text=True)
        except OSError:
            continue
        if r.returncode == 0:
            return str(c), r.stdout.splitlines()[0]
    return None, None


OSMIUM, _osmium_version = _find_osmium()
if OSMIUM is None:
    raise RuntimeError(
        'No working osmium found. Section 2 needs it to clip the PBF. Looked in '
        f'{sys.prefix} and on PATH. Install it with:  '
        'conda install -c conda-forge osmium-tool'
    )

import pandas as pd
import geopandas as gpd
from pyrosm import OSM

from config import (
    STUDY_BOUNDARY_FILE, OSM_PBF_FILE,
    CLIPPED_PBF_FILE, ALL_POIS_FILE, ALL_BUILDINGS_OSM_FILE,
    OUTPUT_DIR, TARGET_CRS,
    POI_EXTRACT_FILTER, POI_EXTRACT_BY_VALUE, POI_BUILDING_ACTIVITY_TAGS, POI_NOT_EXTRACTED,
    POI_EXTRA_ATTRIBUTES, POI_USE_SOURCES,
    POI_EXCLUDE_VALUES, POI_ESCAPE_HATCH_IGNORES, POI_ESCAPE_HATCH_POINTER_VALUES,
    POI_VETO_RULES, POI_INTERIOR_EVIDENCE, POI_NON_ENTERABLE_BUILDINGS,
    POI_BLOCK_EXCEPTIONS, POI_MUST_SURVIVE, DROPPED_POIS_FILE,
    EXPERIMENTAL_DIR,
    EXCLUDE_LIFECYCLE_PREFIXES, EXCLUDE_PLACEHOLDER_USES,
    POI_ANCILLARY_BUILDING_TAGS, POI_MIN_BUILDING_AREA_M2,
    POI_UNIT_TAG_KEYS, POI_SPLIT_AREA_FALLBACK_M2,
    POI_KEEP_TAG_COLS, POI_KEEP_DESC_COLS, POI_KEEP_ADDR_COLS,
    POI_DROP_META_COLS,
    BUILDING_KEEP_COLS, BUILDING_DROP_META_COLS,
)
from lib.checks import (
    require_file, require_non_empty, require_crs, require_unique, count_outside,
)
from lib.schema import fold_tags, osm_key, tidy_names, assert_no_empty_columns

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Root       :', ROOT_DIR)
print('osmium     :', _osmium_version, '|', OSMIUM)
print('Target CRS :', TARGET_CRS)

Root       : C:\Users\Mayur Patel\Documents\GitHub\Capacity_Calculation-pipeline-FINAL
osmium     : osmium version 1.19.1 | C:\Users\Mayur Patel\anaconda3\envs\capacity-final\Library\bin\osmium.exe
Target CRS : EPSG:25832


## 1. Input contract

Both inputs must exist before anything else runs — a missing PBF should fail
here in a second, not twenty minutes into an `osmium` call.

The boundary is dissolved once, in the target CRS, and reused for every
containment report below.

In [2]:
require_file(STUDY_BOUNDARY_FILE, 'study boundary')
require_file(OSM_PBF_FILE, 'OSM regional extract')

boundary = gpd.read_file(STUDY_BOUNDARY_FILE)
require_non_empty(boundary, 'boundary')
print(f'  ..  boundary CRS: {boundary.crs.to_string()}, {len(boundary)} feature(s)')

boundary_target = boundary.to_crs(TARGET_CRS)
boundary_geom   = boundary_target.geometry.union_all()
print(f'  ..  bounds (target CRS): {tuple(round(v) for v in boundary_geom.bounds)}')

  ok  regionalverband_area.gpkg (0.2 MB)
  ok  niedersachsen-260113.osm.pbf (478.2 MB)
  ok  boundary: 9 rows
  ..  boundary CRS: EPSG:25832, 9 feature(s)
  ..  bounds (target CRS): (567880, 5721966, 642410, 5854769)


## 2. Clip the PBF

`osmium` needs the polygon in WGS84 regardless of the boundary file's own CRS.

`-s complete_ways` keeps any way that touches the polygon *whole*, so a building
straddling the border survives intact rather than being cut into an invalid
geometry. The cost is a small overhang outside the boundary, which the
containment reports below quantify.

In [3]:
boundary_wgs = boundary.to_crs(epsg=4326)
geom_wgs     = boundary_wgs.geometry.union_all()

with tempfile.NamedTemporaryFile(mode='w', suffix='.geojson', delete=False) as f:
    json.dump({'type': 'Feature', 'geometry': geom_wgs.__geo_interface__, 'properties': {}}, f)
    poly_file = f.name
print(f'Clip polygon: {geom_wgs.geom_type}, '
      f'{len(geom_wgs.exterior.coords) if geom_wgs.geom_type == "Polygon" else "multi":,} vertices')

# Streamed line by line rather than captured: subprocess output goes to a file
# descriptor and ipykernel only redirects Python-level stdout, so a captured run
# prints NOTHING until it returns and looks like a hang.
print(f'Clipping {OSM_PBF_FILE.stat().st_size / 1e6:,.0f} MB with osmium '
      f'(expect ~5 s warm, up to a minute on a cold cache) ...', flush=True)
t0 = time.perf_counter()
proc = subprocess.Popen(
    [OSMIUM, 'extract',
     '-p', poly_file,
     '-s', 'complete_ways',
     '-o', str(CLIPPED_PBF_FILE),
     '--overwrite',
     str(OSM_PBF_FILE)],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
)
tail = []
for line in proc.stdout:
    line = line.rstrip()
    if line:
        tail.append(line)
        print('   ', line, flush=True)
rc = proc.wait()
if rc != 0:
    raise RuntimeError('osmium extract failed (rc={}):\n{}'.format(rc, '\n'.join(tail[-15:])))

print(f'Clipped PBF: {CLIPPED_PBF_FILE.stat().st_size / 1e6:,.1f} MB '
      f'-> {CLIPPED_PBF_FILE.name}   [{time.perf_counter() - t0:.1f}s]')

Clip polygon: Polygon, 3,299 vertices
Clipping 478 MB with osmium (expect ~5 s warm, up to a minute on a cold cache) ...


Clipped PBF: 57.7 MB -> 01_study_area_clipped.pbf   [4.4s]


## 3. Buildings

A fresh `OSM` object is used per extraction: the reader is stateful, and reusing
one across `get_buildings()` / `get_pois()` has produced empty results.

Two things are removed.

**Non-polygon geometry.** A building is an area. OSM occasionally carries a
`building=*` tag on an unclosed way, which arrives here as a LineString — no
footprint, no area, nothing to match a building against.

**OSM edit metadata and contact details** — `version`, `changeset`, `visible`,
`website`, `phone`. Tags are folded into `tags`; the rest describe the edit
rather than the building and are dropped.

`building:levels` and `height` are kept: they are the only OSM inputs to a
volume.

In [4]:
print('Parsing the clipped PBF for buildings (expect ~40 s) ...', flush=True)
_t0 = time.perf_counter()
osm_bld = OSM(str(CLIPPED_PBF_FILE))
buildings = osm_bld.get_buildings()
print(f'  ..  parsed in {time.perf_counter() - _t0:.1f}s')

require_non_empty(buildings, 'osm_buildings')
buildings = buildings.to_crs(TARGET_CRS)
require_crs(buildings, TARGET_CRS, 'osm_buildings')
print('  ..  raw geometry types:', buildings.geometry.geom_type.value_counts().to_dict())

is_poly = buildings.geometry.geom_type.isin(['Polygon', 'MultiPolygon'])
if (~is_poly).any():
    print(f'  ..  dropped {(~is_poly).sum():,} non-polygon geometries: '
          f'{buildings.loc[~is_poly].geometry.geom_type.value_counts().to_dict()}')
buildings = buildings.loc[is_poly].reset_index(drop=True)
require_non_empty(buildings, 'osm_buildings (polygons)')
print(f'  ..  invalid geometries: {int((~buildings.geometry.is_valid).sum()):,}')

buildings, folded = fold_tags(buildings,
                              keep=BUILDING_KEEP_COLS + ['osm_type', 'id'],
                              meta=BUILDING_DROP_META_COLS)
print(f'  ..  folded {len(folded)} sparse tag columns into `tags`')

buildings = osm_key(buildings, 'bld_id')
buildings = buildings[
    ['bld_id', 'osm_type', 'id']
    + [c for c in BUILDING_KEEP_COLS if c in buildings.columns]
    + ['tags', 'geometry']
].rename(columns={'id': 'osm_id'})
buildings = tidy_names(buildings).reset_index(drop=True)

require_unique(buildings, 'bld_id', 'osm_buildings')
assert_no_empty_columns(buildings, 'osm_buildings')
bld_outside = count_outside(buildings, boundary_geom, 'osm_buildings')
print(f'  ok  buildings prepared: {len(buildings):,} rows, {len(buildings.columns)} columns')

Parsing the clipped PBF for buildings (expect ~40 s) ...


  ..  parsed in 28.1s
  ok  osm_buildings: 509,793 rows


  ok  osm_buildings: CRS EPSG:25832
  ..  raw geometry types: {'Polygon': 509757, 'MultiPolygon': 35, 'LineString': 1}
  ..  dropped 1 non-polygon geometries: {'LineString': 1}


  ok  osm_buildings (polygons): 509,792 rows


  ..  invalid geometries: 0
  ..  folded 0 sparse tag columns into `tags`


  ok  osm_buildings.bld_id: unique and non-null (509,792)


  ok  osm_buildings: 23 columns, none empty


  ..  osm_buildings: 0 of 509,792 rows outside boundary (0.00 %)
  ok  buildings prepared: 509,792 rows, 23 columns


## 4. POIs

**Which keys are extracted is set explicitly**, in `POI_EXTRACT_FILTER`.
`pyrosm`'s default is `amenity`, `shop` and `tourism` and nothing else, which
loses every workplace OSM describes with another key. In this region that is
1,525 `office` features of which only 27 also carry one of the three, plus
~310 `healthcare`-only and ~174 `club`-only features. Offices are occupied
buildings; missing 1,500 of them is a larger error for a capacity pipeline
than any amount of street furniture wrongly kept.

`healthcare` and `club` go through `extra_attributes` rather than the filter:
neither is in `pyrosm`'s tag configuration, so naming them in the filter alone
would match on them without promoting them to columns.

Line geometries are removed. OSM tags a handful of POIs onto ways that are not
areas - a path tagged `tourism=information`, a barrier tagged `amenity=*`. They
are neither a location nor a footprint.

Points and areas both stay. Roughly 36 % of OSM POIs are mapped as areas
(schools, supermarkets and sports centres drawn as their building outline), and
those footprints carry more information than a centroid does.

### Two passes, because mappers use two schemes

The first pass takes every feature with one of the 20 keys in
`POI_EXTRACT_FILTER`, any value. That misses the **Goslar scheme**: an
industrial hall drawn as `building=industrial` + `name=KKF Fels GmbH & Co. KG`
and nothing else, or a business park drawn as `landuse=industrial`. A bare
`building` tag cannot be a candidate key — that would be all 509,792 buildings —
so the second pass, `POI_EXTRACT_BY_VALUE`, takes a few keys **only with listed
values**: buildings with a use value, `man_made` plants, industrial/commercial/
retail land use, stations, power plants, terminals. Section 5 then requires a
*name* on the building-only ones. The keys deliberately left out, with the
reason and the region's counts, are in `POI_NOT_EXTRACTED`, and the cell asserts
that none of them slipped into a filter.

In [5]:
print('Parsing the clipped PBF for POIs (expect ~25 s) ...', flush=True)
_t0 = time.perf_counter()
osm_pois = OSM(str(CLIPPED_PBF_FILE))
pois = osm_pois.get_pois(custom_filter=POI_EXTRACT_FILTER,
                         extra_attributes=POI_EXTRA_ATTRIBUTES)
print(f'  ..  parsed in {time.perf_counter() - _t0:.1f}s: {len(pois):,} features from the {len(POI_EXTRACT_FILTER)} any-value keys')

# The keys must not contradict the record of what is deliberately left out.
_clash = (set(POI_EXTRACT_FILTER) | set(POI_EXTRACT_BY_VALUE)) & set(POI_NOT_EXTRACTED)
if _clash:
    raise AssertionError(f'{sorted(_clash)} are both extracted and listed in POI_NOT_EXTRACTED')

# --- pass 2: keys that count only with listed values --------------------------
print('Second pass for keys that count only with certain values (expect ~40 s) ...', flush=True)
_t1 = time.perf_counter()
osm_extra = OSM(str(CLIPPED_PBF_FILE))
extra = osm_extra.get_data_by_custom_criteria(custom_filter=POI_EXTRACT_BY_VALUE, filter_type='keep',
                                              keep_nodes=True, keep_ways=True, keep_relations=True)
print(f'  ..  parsed in {time.perf_counter() - _t1:.1f}s: {len(extra):,} features')
# pyrosm promotes a key to a column only when its own tag table knows it;
# `man_made` and friends can come back folded inside the `tags` JSON. Pull any
# missing key out of there so the value test below sees every feature.
_missing2 = [k for k in POI_EXTRACT_BY_VALUE if k not in extra.columns]
if _missing2:
    def _from_tags(raw, key):
        if isinstance(raw, str) and raw.strip():
            try:
                return json.loads(raw).get(key)
            except ValueError:
                return None
        return None
    for _k in _missing2:
        extra[_k] = extra['tags'].map(lambda r, _k=_k: _from_tags(r, _k)) if 'tags' in extra.columns else None
    print(f'  ..  {_missing2} were not promoted to columns by pyrosm - read from the tags JSON instead')
# keep only rows where one of the value-filtered keys actually carries an
# accepted value - pyrosm can return relation members and the like alongside
_ok = pd.Series(False, index=extra.index)
for _k, _vals in POI_EXTRACT_BY_VALUE.items():
    _ok |= extra[_k].isin(_vals)
extra = extra.loc[_ok]
print('  ..  per key:', {k: int(extra[k].isin(v).sum()) for k, v in POI_EXTRACT_BY_VALUE.items()})
_key1 = pois['osm_type'].astype(str) + '/' + pois['id'].astype(str)
_key2 = extra['osm_type'].astype(str) + '/' + extra['id'].astype(str)
extra = extra.loc[~_key2.isin(set(_key1))]
n_pass2 = len(extra)
print(f'  ..  {n_pass2:,} of them are new - not already found by the first pass')
if extra.crs != pois.crs:
    extra = extra.to_crs(pois.crs)
pois = gpd.GeoDataFrame(pd.concat([pois, extra], ignore_index=True), geometry='geometry', crs=pois.crs)

require_non_empty(pois, 'pois')
_missing = [k for k in list(POI_EXTRACT_FILTER) + list(POI_EXTRACT_BY_VALUE) + POI_EXTRA_ATTRIBUTES
            if k not in pois.columns]
if _missing:
    raise AssertionError(
        f'pyrosm returned no column for {_missing}. The extraction filter asked '
        'for these keys, so a missing column means the filter was ignored and '
        'every feature carrying only that key is absent from the layer.'
    )
print('  ..  extracted per key:',
      {k: int(pois[k].notna().sum())
       for k in list(POI_EXTRACT_FILTER) + list(POI_EXTRACT_BY_VALUE) + POI_EXTRA_ATTRIBUTES})
n_pois_raw = len(pois)
pois = pois.to_crs(TARGET_CRS)
require_crs(pois, TARGET_CRS, 'pois')
print('  ..  raw geometry types:', pois.geometry.geom_type.value_counts().to_dict())

is_line = pois.geometry.geom_type.isin(['LineString', 'MultiLineString'])
if is_line.any():
    print(f'  ..  dropped {is_line.sum():,} line geometries')
pois = pois.loc[~is_line].reset_index(drop=True)
require_non_empty(pois, 'pois (points + areas)')
n_pois_geom = len(pois)

Parsing the clipped PBF for POIs (expect ~25 s) ...


  ..  parsed in 27.1s: 83,393 features from the 20 any-value keys
Second pass for keys that count only with certain values (expect ~40 s) ...


  ..  parsed in 23.6s: 11,437 features
  ..  ['man_made'] were not promoted to columns by pyrosm - read from the tags JSON instead
  ..  per key: {'building': 8972, 'man_made': 547, 'landuse': 1847, 'railway': 52, 'power': 85, 'aeroway': 7}
  ..  9,386 of them are new - not already found by the first pass


  ok  pois: 92,779 rows
  ..  extracted per key: {'amenity': 52726, 'shop': 6175, 'tourism': 9639, 'office': 1525, 'craft': 644, 'leisure': 9289, 'healthcare': 1216, 'club': 192, 'sport': 4017, 'historic': 2820, 'religion': 1116, 'social_facility': 369, 'government': 89, 'industrial': 83, 'university': 70, 'military': 57, 'education': 31, 'company': 19, 'trade': 19, 'school': 9, 'building': 15272, 'man_made': 576, 'landuse': 2126, 'railway': 55, 'power': 101, 'aeroway': 8}


  ok  pois: CRS EPSG:25832
  ..  raw geometry types: {'Point': 49533, 'Polygon': 42572, 'MultiLineString': 420, 'LineString': 174, 'MultiPolygon': 80}
  ..  dropped 594 line geometries
  ok  pois (points + areas): 92,185 rows


## 5. Keep only POIs that describe an indoor activity

A POI is useful downstream only if it names something happening **inside** a
building. Street furniture, open-air infrastructure and parking do not, and
would otherwise be assigned to whichever building they happen to sit in.

### The named-building rule

A feature that the second pass brought in on its `building` value alone —
no amenity, no shop, no office, nothing but `building=industrial` — is a POI
only if it is **named**. The name is what says a business is there; an unnamed
industrial hall is just a hall, and its use still reaches the ALKIS building
through `osm_twin_tag` in step 04. The other second-pass keys count named or
not: an unnamed `landuse=industrial` estate still tells every building inside
it where it stands, an unnamed sewage works is still a workplace.

### Every rule is a block-list

That is a deliberate change. An earlier version used **allow-lists** for
`tourism`, `information` and `leisure`, and the difference matters:

```
a block-list drops what you have judged
an allow-list drops everything nobody thought to name
```

The second is a hard drop by omission, and it was doing real damage. The
`tourism` allow-list permitted 8 values, so `zoo`, `attraction`, `camp_site` and
`caravan_site` were deleted for not being on it — **86 % of that rule's drops
were wrong**. `leisure` deleted water parks, marinas, golf courses and riding
stables the same way.

Downstream classification is done by an LLM, which handles an odd tag far better
than this filter handles an unlisted one. So the bar here is **"definitely not a
visitable interior"**, not "recognised venue". `POI_EXCLUDE_VALUES` in
`config.py` holds all five block-lists in one place.

### The escape hatch, and what may not use it

A rule judges its key **only when that key is the only thing describing the
feature**. `amenity=fuel` + `building=roof` is a petrol station, not a roof —
and 25 named petrol stations were being deleted for exactly that.

Two *keys* never count as "something else describing the feature", because on
their own they describe nothing:

* **`building`** says there is a structure, not what happens in it. Counting it
  readmitted 58 outdoor features — pitches, stadiums, playgrounds — on the
  strength of a bare `building=yes`.
* **`sport`** qualifies a `leisure` feature. `sport=soccer` on a pitch does not
  put the pitch indoors, yet counting it let **2,753 pitches, 167 running tracks
  and 54 fitness stations** through the `leisure` rule. Features carrying *only*
  `sport` — climbing walls, shooting ranges — still survive: the final lifecycle
  rule needs one informative key and `sport` counts there.

One *value* never counts either. `tourism=information` is a **pointer** to the
`information` key, not a use of its own. Before `POI_ESCAPE_HATCH_POINTER_VALUES`
existed, every guidepost was rescued from the `information` rule by the very tag
that sent it there: the rule dropped 0 rows in this region and **6,489
guideposts, boards, maps and route markers — 19 % of the layer — reached the
output**.

### The veto: interior evidence beats the block-list

For the `information`, `leisure` and `historic` rules a block-listed row is kept
anyway when its full tag set — promoted columns plus the folded `tags` JSON —
shows an enterable structure: a `building` value outside
`POI_NON_ENTERABLE_BUILDINGS`, `building:levels`, `indoor=yes`, and for the
`information` rule also `opening_hours` or a house number, which a signpost never
has and a staffed info point does. The veto is *not* applied to the `amenity`
rule: there it would readmit 2,148 rows, mostly cash machines open 24/7, car
parks and recycling containers.

Measured on this region the veto keeps about 40 rows, and they are the right
ones: a shooting range mapped as a `pitch` with `building=yes`, a "Kalthalle"
pitch that is a `building=sports_hall`, an info terminal with a house number and
opening hours. After it, **no row dropped by these rules carries a building
tag**.

Names are deliberately *not* evidence. Of 5,006 named rows these rules drop,
only 70 have a kept POI of the same name within 100 m, and the business-looking
names are boards *about* a museum or a former pharmacy, not the venue itself.

### Qualifier exceptions

`amenity=recycling` covers 1,855 container stations and 41 staffed recycling
centres (Wertstoffhöfe), and only `recycling_type=centre` tells them apart.
`POI_BLOCK_EXCEPTIONS` keeps a block-listed value when such a qualifier is
present.

### Auditability

Every dropped POI is written to `01_dropped_pois.gpkg` with a `dropped_by`
column naming the rule that removed it, and every kept row that a veto or an
exception saved carries the reason in `rescued_by`. The cell prints which tag
*values* and which *names* each rule took out. Two guards run after the rules:
a rule that matched rows but dropped and rescued none has been defeated by the
escape hatch and fails the cell, and every value in `POI_MUST_SURVIVE` — the
tourist offices — must still be present afterwards.


In [6]:
import json as _json


def _tokens(v):
    """OSM allows `shop=bakery;cafe`. Judge every token, not the raw string."""
    if pd.isna(v):
        return []
    return [t.strip() for t in str(v).replace(',', ';').split(';') if t.strip()]


def _live(t):
    """A token that names something: not a lifecycle prefix, not a placeholder."""
    return bool(t) and not t.startswith(EXCLUDE_LIFECYCLE_PREFIXES) and t not in EXCLUDE_PLACEHOLDER_USES


def is_informative(v):
    """False for a null, a lifecycle-prefixed value or a placeholder."""
    return any(_live(t) for t in _tokens(v))


def _all_tokens_excluded(s, blocked):
    """True only when EVERY token is on the block-list. `shop=bakery;kiosk`
    survives a rule that blocks only `kiosk`."""
    blocked = set(blocked)
    return s.map(lambda v: bool(_tokens(v)) and all(t in blocked for t in _tokens(v)))


def _vouches(col, s):
    """Can column `col` rescue a row from a rule? Informative, and not a pointer
    value: `tourism=information` only says "look at the information key", so it
    may not vouch for the guidepost that key describes."""
    pointers = set(POI_ESCAPE_HATCH_POINTER_VALUES.get(col, ()))
    return s.map(lambda v: any(_live(t) and t not in pointers for t in _tokens(v)))


def _has_other_use(d, key):
    """Is any use-bearing key OTHER than `key` describing this feature?

    This is the escape hatch. Keys in POI_ESCAPE_HATCH_IGNORES never count:
    `building` says there is a structure, not what happens in it, and `sport`
    only qualifies a `leisure` feature.
    """
    ignore = {key} | set(POI_ESCAPE_HATCH_IGNORES)
    cols = [c for c in POI_USE_SOURCES if c not in ignore and c in d.columns]
    if not cols:
        return pd.Series(False, index=d.index)
    return pd.concat([_vouches(c, d[c]) for c in cols], axis=1).any(axis=1)


# pyrosm only returns `information` when something in the extract carries it.
for _c in ['information'] + POI_USE_SOURCES:
    if _c not in pois.columns:
        pois[_c] = pd.NA

# --- the FULL tag set of every row ------------------------------------------
# pyrosm promotes `building` and `opening_hours` to columns but leaves `indoor`
# and `recycling_type` inside the `tags` JSON. The veto and the exceptions have
# to see both, so each row gets one dict of everything it carries. Built column
# by column over the non-null cells only, which is ~600k values, not 82k x 130.
_NOT_TAGS = set(POI_DROP_META_COLS) | {'timestamp', 'id', 'osm_type', 'tags', pois.geometry.name}
tag_sets = {i: {} for i in pois.index}
for _c in [c for c in pois.columns if c not in _NOT_TAGS]:
    for _i, _v in pois[_c].dropna().items():
        if str(_v) != '':
            tag_sets[_i][_c] = str(_v)
for _i, _raw in pois['tags'].items():
    if isinstance(_raw, str) and _raw.strip():
        try:
            for _k, _v in _json.loads(_raw).items():
                tag_sets[_i].setdefault(_k, str(_v))
        except ValueError:
            pass


def interior_evidence(tags, key):
    """The tags that say "there is an enterable structure here", or [] if none.

    `building` counts unless its value is in POI_NON_ENTERABLE_BUILDINGS;
    `indoor` counts only as `indoor=yes`; `opening_hours` counts unless it is
    `24/7`, which is what cash machines, playgrounds and fitness stations carry;
    every other listed tag counts by being present.
    """
    ev = []
    for k in POI_INTERIOR_EVIDENCE['*'] + POI_INTERIOR_EVIDENCE.get(key, ()):
        v = tags.get(k)
        if v is None:
            continue
        if k == 'building':
            if v not in POI_NON_ENTERABLE_BUILDINGS:
                ev.append(f'building={v}')
        elif k == 'indoor':
            if v == 'yes':
                ev.append('indoor=yes')
        elif k == 'opening_hours':
            if v.strip() != '24/7':
                ev.append('opening_hours')
        else:
            ev.append(k)
    return ev


# None, not pd.NA: pyogrio writes None as NULL but pd.NA as the string '<NA>',
# which would make `rescued_by IS NOT NULL` match every row in QGIS.
pois['rescued_by'] = pd.Series(None, index=pois.index, dtype='object')
_must_survive_before = {k: int(pois[k].isin(v).sum()) for k, v in POI_MUST_SURVIVE.items()}

n_before = len(pois)
dropped_frames = []
rule_stats = {}

# --- the named-building rule ------------------------------------------------
# Only accepted key is `building` -> needs a name and a use value.
_acc = [c for c in POI_USE_SOURCES if c in pois.columns and c != 'building']
_other = pd.concat([pois[c].map(is_informative) for c in _acc], axis=1).any(axis=1)
_bld_only = pois['building'].notna() & ~_other
_bld_bad = _bld_only & (pois['name'].isna() | ~pois['building'].isin(POI_BUILDING_ACTIVITY_TAGS))
gone = pois.loc[_bld_bad]
print(f'  {"building":<14} -{len(gone):>7,}   (building-only features without a name or a use value; '
      f'{int((_bld_only & ~_bld_bad).sum()):,} named ones with a use stay)')
if len(gone):
    g = gone.copy()
    g['dropped_by'] = 'building without name'
    dropped_frames.append(g)
    print(f'  {"":<14}  by value: ' + ', '.join(f'{k} {v:,}' for k, v in gone['building'].value_counts().head(8).items()))
pois = pois.loc[~_bld_bad]
print()
print(f'{n_before:,} POIs in, applying {len(POI_EXCLUDE_VALUES)} block-list rules ...')
print()
# Rules run one after another on the SHRINKING frame, so a later rule's counts
# never include rows an earlier rule already removed.
for key, blocked in POI_EXCLUDE_VALUES.items():
    matched = _all_tokens_excluded(pois[key], blocked)         # the value is on the list ...
    hit = matched & ~_has_other_use(pois, key)                # ... and nothing else vouches
    reason = pd.Series(pd.NA, index=pois.index, dtype='object')

    # qualifier exceptions: a listed value that a second tag turns into a site
    for (_k, _val), _quals in POI_BLOCK_EXCEPTIONS.items():
        if _k != key:
            continue
        cand = hit & pois[key].map(lambda v: _val in _tokens(v))
        for i in cand[cand].index:
            why = [f'{q}={tag_sets[i].get(q)}' for q, ok in _quals.items() if tag_sets[i].get(q) in ok]
            if why:
                reason[i] = ', '.join(why)

    # the veto: interior evidence beats the block-list
    if key in POI_VETO_RULES:
        for i in hit[hit].index:
            if pd.isna(reason[i]):
                ev = interior_evidence(tag_sets[i], key)
                if ev:
                    reason[i] = ', '.join(ev)

    rescued = hit & reason.notna()
    drop = hit & ~rescued
    pois.loc[rescued, 'rescued_by'] = reason[rescued]
    rule_stats[key] = {'matched': int(matched.sum()), 'dropped': int(drop.sum()),
                       'rescued': int(rescued.sum())}

    gone = pois.loc[drop]
    print(f'  {key:<14} -{len(gone):>7,}   rescued {int(rescued.sum()):>4,}')
    if len(gone):
        g = gone.copy()
        g['dropped_by'] = key
        dropped_frames.append(g)
        # WHICH values and WHICH names did it remove? This is the transparency
        # that makes the block-lists auditable rather than a number to trust.
        vc = gone[key].value_counts().head(8)
        for v, n in vc.items():
            print(f'  {"":<14}  {str(v)[:34]:<34} {n:>7,}')
        if gone[key].nunique() > 8:
            print(f'  {"":<14}  ... and {gone[key].nunique() - 8} more values')
        if gone['name'].notna().any():
            names = ', '.join(f'{n} ({c})' for n, c in gone['name'].value_counts().head(5).items())
            print(f'  {"":<14}  names: {names}')
    pois = pois.loc[~drop]

# lifecycle / placeholder: nothing informative is left on the row
keep = pd.concat([pois[c].map(is_informative) for c in POI_USE_SOURCES if c in pois.columns],
                 axis=1).any(axis=1)
gone = pois.loc[~keep]
print(f'  {"lifecycle":<14} -{len(gone):>7,}')
if len(gone):
    g = gone.copy()
    g['dropped_by'] = 'lifecycle / placeholder'
    dropped_frames.append(g)
pois = pois.loc[keep].reset_index(drop=True)
require_non_empty(pois, 'pois (filtered)')
n_rescued = int(pois['rescued_by'].notna().sum())
print()
print(f'  ok  kept {len(pois):,} of {n_before:,} ({100 * len(pois) / n_before:.1f} %), '
      f'{n_rescued:,} of them rescued by evidence or exception')

# --- guard 1: a rule that matches rows but removes none has been defeated ------
# This is exactly how the `information` rule failed: `tourism=information`
# vouched for every guidepost, so the rule matched thousands and dropped 0.
# Rescues count as the rule working - the veto saw the rows and kept them.
_defeated = [k for k, s in rule_stats.items()
             if s['matched'] >= 20 and s['dropped'] + s['rescued'] == 0]
if _defeated:
    raise AssertionError(
        f'rules {_defeated} matched rows but dropped and rescued none: a key in '
        'POI_USE_SOURCES is vouching for the very rows the rule exists to judge. '
        'Add the key to POI_ESCAPE_HATCH_IGNORES or the value to '
        'POI_ESCAPE_HATCH_POINTER_VALUES.'
    )

# --- guard 2: values that must never be filtered out ---------------------------
for _key, _vals in POI_MUST_SURVIVE.items():
    _want, _have = _must_survive_before[_key], int(pois[_key].isin(_vals).sum())
    if _have != _want:
        raise AssertionError(
            f'{_key} in {list(_vals)}: {_want:,} rows before the filter, {_have:,} after. '
            'A block-list or the escape hatch is removing rows that must survive.'
        )
    print(f'  ok  {_key} in {list(_vals)}: all {_have:,} survived the filter')

# --- write the dropped rows so the filter can be checked by eye --------------
if dropped_frames:
    dropped = gpd.GeoDataFrame(pd.concat(dropped_frames, ignore_index=True),
                               crs=pois.crs)
    _keep = ['dropped_by', 'name', 'operator', 'information'] + \
            [c for c in POI_USE_SOURCES if c in dropped.columns] + \
            ['osm_type', 'id', 'geometry']
    dropped = dropped[[c for c in dict.fromkeys(_keep) if c in dropped.columns]]
    EXPERIMENTAL_DIR.mkdir(parents=True, exist_ok=True)
    if DROPPED_POIS_FILE.exists():
        DROPPED_POIS_FILE.unlink()
    dropped.to_file(DROPPED_POIS_FILE, layer='dropped', driver='GPKG')
    _named = int(dropped['name'].notna().sum()) if 'name' in dropped.columns else 0
    print(f'  ..  {len(dropped):,} dropped rows -> {DROPPED_POIS_FILE.name} '
          f'({_named:,} of them named - style by `dropped_by` in QGIS)')


  building       -  5,482   (building-only features without a name or a use value; 1,499 named ones with a use stay)
                  by value: industrial 1,255, retail 1,014, commercial 931, school 627, office 533, warehouse 260, kindergarten 183, hotel 93

92,185 POIs in, applying 5 block-list rules ...



  amenity        - 43,325   rescued   29
                  parking                             10,241
                  bench                                9,728
                  parking_space                        5,882
                  waste_basket                         3,340
                  bicycle_parking                      2,907
                  hunting_stand                        2,335
                  recycling                            1,855
                  shelter                              1,501
                  ... and 60 more values
                  names: Kundenparkplatz (16), Zigarettenautomat (12), Stromtankstelle BS Energy (12), ubitricity (12), WC Herren (10)


  tourism        -  1,061   rescued    0
                  picnic_site                            385
                  viewpoint                              360
                  artwork                                316
                  names: Brockenblick (7), Steinway Flügel (3), Fischotter (2), Galgenberg (2), Januskopf (2)


  information    -  6,523   rescued    5
                  guidepost                            3,267
                  board                                2,491
                  map                                    532
                  route_marker                           180
                  hikingmap                               33
                  tactile_model                            7
                  tactile_map                              4
                  depature_board                           4
                  ... and 3 more values
                  names: Radwanderregion Südheide Gifhorn (40), Naturlehrpfad (34), Naturschutzgebiet Allertal zwischen Gifhorn und Wolfsburg (25), Friedenspfad (23), Freizeitradwegenetz Gemeinde Sassenburg (21)


  leisure        -  7,148   rescued   34
                  pitch                                2,848
                  playground                           1,946
                  park                                   646
                  picnic_table                           632
                  garden                                 448
                  track                                  203
                  nature_reserve                         113
                  fitness_station                         76
                  ... and 8 more values
                  names: A-Platz (19), Bolzplatz (15), Kinderspielplatz (14), B-Platz (14), TC Fallersleben (9)


  historic       -  1,643   rescued    1
                  memorial                             1,227
                  boundary_stone                         303
                  milestone                               63
                  stone                                   22
                  tomb                                    18
                  wayside_cross                            9
                  pillory                                  1
                  names: OD Stein (23), Kriegerdenkmal (15), Ehrenmal (11), Erzbrocken (9), Weltkriegsdenkmal (7)


  lifecycle      -      4
  ok  pois (filtered): 26,999 rows

  ok  kept 26,999 of 92,185 (29.3 %), 69 of them rescued by evidence or exception
  ok  information in ['office', 'visitor_centre']: all 46 survived the filter


  ..  65,186 dropped rows -> 01_dropped_pois.gpkg (7,408 of them named - style by `dropped_by` in QGIS)


## 6. `poi_role` — how each POI can be joined to a building

Area POIs are not one kind of thing, and joining them uniformly is wrong either
way:

* **collapsing areas to points** reduces a 370,000 m² university campus to one
  arbitrary shed, and can drop a centroid into an unrelated neighbour;
* **joining every area to every building it touches** pulls in neighbours whose
  wall merely abuts the site boundary — measured at ~6 % spurious pairs.

| `poi_role` | Meaning | Join by |
|---|---|---|
| `point` | node POI | the building containing it |
| `footprint` | area covering one building, or none in OSM | largest intersection area, 1:1 |
| `site` | area covering 2+ real buildings (school, hospital, campus) | all contained buildings, carrying `poi_id` as a site key |

`site` marks an area covering several buildings — a hospital campus really is
several hospital buildings — and `poi_id` identifies which buildings belong to
the same site.

Counting raw footprints would make a school with three garages look like a
four-building site, so ancillary structures and anything below
`POI_MIN_BUILDING_AREA_M2` are ignored. Buildings are matched on
`representative_point` **within** the area, not `intersects`, for the reason
above.

An area with **no** building inside it is classed `footprint`, not discarded.
That emptiness is a fact about OSM's coverage, not about reality. `n_osm_bld`
records what was found and is a hint, not a verdict.

In [7]:
areas_mask = pois.geometry.geom_type.isin(['Polygon', 'MultiPolygon'])
areas = pois.loc[areas_mask, ['geometry']].copy()
areas['_aix'] = range(len(areas))

bld_real = buildings.loc[
    ~buildings['building'].isin(POI_ANCILLARY_BUILDING_TAGS)
    & (buildings.geometry.area >= POI_MIN_BUILDING_AREA_M2)
]
print(f'  ..  buildings counted as real: {len(bld_real):,} of {len(buildings):,} '
      f'({100 * len(bld_real) / len(buildings):.1f} %)')

if areas.empty:
    n_bld = pd.Series(dtype='int64')
else:
    pts = gpd.GeoDataFrame(geometry=bld_real.geometry.representative_point(),
                           crs=bld_real.crs)
    hit = gpd.sjoin(pts, areas, how='inner', predicate='within')
    n_bld = areas['_aix'].map(hit.groupby('_aix').size()).fillna(0).astype(int)

pois['n_osm_bld'] = 0
pois.loc[areas.index, 'n_osm_bld'] = n_bld.values
pois['geom_kind'] = pois.geometry.geom_type.map(
    {'Point': 'point', 'Polygon': 'area', 'MultiPolygon': 'area'})
pois['area_m2'] = 0.0
pois.loc[areas.index, 'area_m2'] = pois.loc[areas.index].geometry.area.round(1)

pois['poi_role'] = 'point'
pois.loc[areas_mask, 'poi_role'] = 'footprint'
pois.loc[areas_mask & (pois['n_osm_bld'] >= 2), 'poi_role'] = 'site'

print()
print('  ..  poi_role:')
for role, cnt in pois['poi_role'].value_counts().items():
    print(f'        {role:<10} {cnt:>7,}')
print(f'  ..  areas with no OSM building: '
      f'{int((areas_mask & (pois["n_osm_bld"] == 0)).sum()):,}')

  ..  buildings counted as real: 386,014 of 509,792 (75.7 %)



  ..  poi_role:
        point       12,838
        footprint   12,098
        site         2,063
  ..  areas with no OSM building: 2,493


## 7. `poi_use` and the schema

**`poi_use` is a convenience, not a source of truth.** It collapses the
use-bearing columns into one readable string so the layer can be styled and
eyeballed in QGIS.

It is deliberately **not** what downstream classification reads. Every tag a POI
carries survives in this layer — verified against `osmium` on 31 POIs, **606 of
606 tags reachable** either as a real column or inside the `tags` JSON — so the
LLM step gets the full picture and `poi_use` discarding detail costs nothing.

**The order of `POI_USE_SOURCES` no longer carries meaning.** It used to be a
priority ranking for picking one winner, which invited unanswerable arguments
about whether `sport` outranks `leisure`. The filter above needs only the *set*
— "is any use-bearing tag informative?" — and `poi_use` needs only *a* readable
answer, not the *best* one. First informative value still wins; which one that
is is now an implementation detail.

Two behaviours worth keeping:

* a value that is not informative — lifecycle-prefixed or a placeholder — is
  **skipped rather than taken**, so `amenity=disused:restaurant` + `shop=bakery`
  gives `bakery` rather than a restaurant that closed;
* `office=yes` and `shop=yes` resolve to the **key**, not the literal `yes` —
  that a use exists but is unspecified is worth recording; `yes` alone is not.

Section 5 has already dropped the rows where *nothing* informative was left, so
`poi_use` cannot come out null.

The schema reduction is the same lossless fold used for the buildings: every
dropped tag column is folded into `tags`, which is where `capacity`,
`capacity:beds`, `beds`, `seats` and `level` live.

In [8]:
pois['poi_use'] = pd.Series(pd.NA, index=pois.index, dtype='object')
pois['poi_use_tag'] = pd.Series(pd.NA, index=pois.index, dtype='object')
for _col in POI_USE_SOURCES:
    if _col not in pois.columns:
        continue
    # `is_informative`, not `notna`: a dead or placeholder value must not win
    # the slot and block a lower-priority column that has a real answer.
    _fill = pois['poi_use'].isna() & pois[_col].map(is_informative)
    # `office=yes` says a use exists but not which one. Resolving it to the KEY
    # rather than the literal 'yes' keeps the one fact it does carry - this is
    # an office - instead of a value that means nothing on its own. Widening
    # the extraction filter would otherwise have grown `poi_use='yes'` from 43
    # rows to 125.
    _val = pois.loc[_fill, _col]
    pois.loc[_fill, 'poi_use'] = _val.where(_val.astype(str).str.strip() != 'yes', _col)
    pois.loc[_fill, 'poi_use_tag'] = _col
print('  ..  poi_use resolved from:', pois['poi_use_tag'].value_counts().to_dict())
_n_null = int(pois['poi_use'].isna().sum())
if _n_null:
    raise AssertionError(
        f'{_n_null:,} POIs have no informative use tag, but section 5 should '
        'already have dropped every such row. The two rules have drifted apart.'
    )
print(f'  ..  poi_use null: {_n_null:,} of {len(pois):,}')

derived = ['poi_use', 'poi_use_tag', 'poi_role', 'geom_kind', 'area_m2', 'n_osm_bld',
           'rescued_by']
keep = (POI_KEEP_DESC_COLS + POI_KEEP_TAG_COLS + POI_KEEP_ADDR_COLS
        + derived + ['osm_type', 'id'])

pois, folded = fold_tags(pois, keep=keep, meta=POI_DROP_META_COLS)
print(f'  ..  folded {len(folded)} sparse tag columns into `tags`')

pois = osm_key(pois, 'poi_id')
pois = pois[
    ['poi_id', 'osm_type', 'id'] + derived
    + [c for c in POI_KEEP_DESC_COLS if c in pois.columns]
    + [c for c in POI_KEEP_TAG_COLS if c in pois.columns]
    + [c for c in POI_KEEP_ADDR_COLS if c in pois.columns]
    + ['tags', 'geometry']
].rename(columns={'id': 'osm_id'})
pois = tidy_names(pois).reset_index(drop=True)

require_unique(pois, 'poi_id', 'pois')
assert_no_empty_columns(pois, 'pois', allow_empty=['rescued_by'])
pois_outside = count_outside(pois, boundary_geom, 'pois')
print(f'  ok  POIs prepared: {len(pois):,} rows, {len(pois.columns)} columns')

  ..  poi_use resolved from: {'amenity': 9359, 'shop': 6077, 'tourism': 1854, 'landuse': 1698, 'building': 1499, 'office': 1498, 'leisure': 1360, 'sport': 870, 'historic': 651, 'man_made': 576, 'craft': 574, 'healthcare': 280, 'religion': 171, 'club': 170, 'industrial': 78, 'power': 70, 'university': 69, 'military': 57, 'railway': 54, 'social_facility': 12, 'school': 7, 'aeroway': 7, 'company': 4, 'education': 2, 'trade': 1, 'government': 1}
  ..  poi_use null: 0 of 26,999


  ..  folded 126 sparse tag columns into `tags`
  ok  pois.poi_id: unique and non-null (26,999)
  ok  pois: 32 columns, none empty


  ..  pois: 1 of 26,999 rows outside boundary (0.00 %)
  ok  POIs prepared: 26,999 rows, 32 columns


## 8. Nesting — a POI inside another POI

Two things in OSM are one-to-many, and both matter for how demand is placed
later:

* **a mall** is one building holding many shops — Schloss-Arkaden is one
  footprint with 128 POIs inside it, 125 of them drawn as indoor unit outlines;
* **a campus** is one area holding many buildings — TU Clausthal is one polygon
  over 73 POIs that each sit in their own building.

Three columns record this. The rule and the measurements behind it are in
`docs/nested-poi-area-split.md`.

**`poi_parent_id`** — the smallest POI polygon that contains the POI's
representative point, other than itself and larger than it — a hall standing on
an industrial estate must not become the estate's parent. A shop in a mall on a campus points at
the mall; the mall points at the campus.

**Indoor units can never be parents.** A polygon with an `indoor` or `level`
*key* and no `building` tag is a room outline, and rooms on different floors
overlap in plan — so without this rule a first-floor shop becomes the parent of
the ground-floor shop below it (549 wrong parents when measured). It has to be
the *key*: the Schloss-Arkaden building itself carries `surveillance=indoor` as
a value.

**`poi_role = 'unit'`** — a point or footprint POI whose parent carries a
`building=*` tag. The parent *is* one building (a mall, a supermarket with a
bakery inside, a hotel, a town hall), so its children share that building's
volume. Children of an area parent (campus, outlet village, zoo, holiday park)
are buildings of their own and keep their role; a `site` child stays a `site`.
The parent's `building` tag decides this, not its `poi_role`: Schloss-Arkaden is
classed `site` above because two OSM buildings fall inside it, yet it is one
building with `building=retail`.

**`split_area_m2`** — units only. A polygon unit uses its own area; a point unit
takes the median area of its polygon siblings under the same parent; when no
sibling has an area every point gets `POI_SPLIT_AREA_FALLBACK_M2`, which makes
the split even. Points carry `area_m2 = 0.0`, not null, so the test is on
`geom_kind`. Unit outlines are indoor rooms across several levels — at
Schloss-Arkaden they sum to 154 % of the footprint — so this is only ever a
*share within the parent*, never an absolute area. The parent itself gets no
share, or the mall would be counted as the whole and as its tenants.


In [9]:
import json as _json


def _tag_keys(raw):
    if isinstance(raw, str) and raw.strip():
        try:
            return set(_json.loads(raw))
        except ValueError:
            return set()
    return set()


# --- 8a. indoor units: a level/indoor KEY and no building tag -----------------
_keys = pois['tags'].map(_tag_keys)
_is_area = pois['geom_kind'] == 'area'
_has_level = _keys.map(lambda k: any(t in k for t in POI_UNIT_TAG_KEYS))
_has_building = pois['building'].notna()
is_unit_poly = _is_area & _has_level & ~_has_building
print(f'  ..  {int(is_unit_poly.sum()):,} area POIs are indoor units '
      f'(a {"/".join(POI_UNIT_TAG_KEYS)} key, no building tag) - they can never be parents')
print(f'      {int((~_is_area & _has_level).sum()):,} point POIs carry a level key too - shop nodes inside malls')

# --- 8b. the parent: smallest non-unit polygon containing the POI, not itself ---
_cand = pois.loc[_is_area & ~is_unit_poly, ['poi_id', 'geometry']].copy()
_cand['_parea'] = _cand.geometry.area
_cand['_pbld'] = _has_building[_cand.index].to_numpy()
_rep = gpd.GeoDataFrame({'poi_id': pois['poi_id']},
                        geometry=pois.geometry.representative_point(), crs=pois.crs)
_hit = gpd.sjoin(_rep, _cand.rename(columns={'poi_id': 'poi_parent_id'}),
                 how='inner', predicate='within')
_hit = _hit[_hit['poi_id'] != _hit['poi_parent_id']]
# A parent must be LARGER than its child. A site's representative point can fall
# inside a hall standing on that site, and the hall would otherwise become the
# parent of the estate around it.
_child_area = pois.set_index('poi_id').geometry.area
_hit = _hit[_hit['_parea'] > _hit['poi_id'].map(_child_area).fillna(0.0).to_numpy()]
_hit = _hit.sort_values('_parea').drop_duplicates('poi_id', keep='first')   # smallest container wins
_par = _hit.set_index('poi_id')
pois['poi_parent_id'] = pois['poi_id'].map(_par['poi_parent_id'])
_parent_is_bld = pois['poi_id'].map(_par['_pbld']).fillna(False).astype(bool)
_nested = pois['poi_parent_id'].notna()
n_nested = int(_nested.sum())
print(f'  ..  {n_nested:,} POIs ({100 * n_nested / len(pois):.1f} %) sit inside another POI polygon, '
      f'under {pois["poi_parent_id"].nunique():,} parents; '
      f'{int(pois["poi_parent_id"].isin(pois.loc[_nested, "poi_id"]).sum()):,} of them under a parent '
      'that is itself nested (shop in a mall on a campus)')

# --- 8c. unit or child of an area: the parent's building tag decides -----------
_unit = _nested & _parent_is_bld & pois['poi_role'].isin(['point', 'footprint'])
_site_kid_of_bld = _nested & _parent_is_bld & (pois['poi_role'] == 'site')
pois.loc[_unit, 'poi_role'] = 'unit'
print(f'  ..  {int(_unit.sum()):,} units under {pois.loc[_unit, "poi_parent_id"].nunique():,} building parents; '
      f'{int((_nested & ~_parent_is_bld).sum()):,} children of area parents keep their role'
      + (f'; {int(_site_kid_of_bld.sum())} site POIs inside a building polygon stay sites' if _site_kid_of_bld.any() else ''))
print('  ..  poi_role now:', pois['poi_role'].value_counts().to_dict())

# --- 8d. split_area_m2 for units ---------------------------------------------------
pois['split_area_m2'] = pd.Series(pd.NA, index=pois.index, dtype='Float64')
_u = pois.loc[_unit, ['poi_parent_id', 'geom_kind', 'area_m2']]
_own = _u['geom_kind'] == 'area'
pois.loc[_u.index[_own], 'split_area_m2'] = _u.loc[_own, 'area_m2'].to_numpy()
_sib_med = _u.loc[_own].groupby('poi_parent_id')['area_m2'].median()
_pts = _u.loc[~_own, 'poi_parent_id']
_has_sib = _pts.isin(_sib_med.index)
pois.loc[_u.index[~_own], 'split_area_m2'] = _pts.map(_sib_med).fillna(POI_SPLIT_AREA_FALLBACK_M2).to_numpy()
_kind = _u.groupby('poi_parent_id')['geom_kind'].agg(
    lambda s: 'all points' if (s == 'point').all() else ('all polygons' if (s == 'area').all() else 'mixed'))
print(f'  ..  split_area_m2 on all {int(pois["split_area_m2"].notna().sum()):,} units: '
      f'{int(_own.sum()):,} own area, {int(_has_sib.sum()):,} sibling median, '
      f'{int((~_has_sib).sum()):,} fallback {POI_SPLIT_AREA_FALLBACK_M2} (even split)')
print('      building parents by child geometry:', _kind.value_counts().to_dict())
if int(pois['split_area_m2'].notna().sum()) != int(_unit.sum()):
    raise AssertionError('every unit must carry a split_area_m2 and nothing else may')
_tot = pois.loc[_unit].groupby('poi_parent_id')['split_area_m2'].sum()
if (_tot <= 0).any():
    raise AssertionError('a building parent has a zero split-area total - its shares would divide by zero')

# --- 8e. the biggest parents, for the eye -----------------------------------------
_names = pois.set_index('poi_id')['name']
_top = (pois.loc[_nested].groupby('poi_parent_id')
        .agg(n=('poi_id', 'size'), units=('poi_role', lambda s: int((s == 'unit').sum())),
             polys=('geom_kind', lambda s: int((s == 'area').sum())))
        .sort_values('n', ascending=False).head(10))
print('  ..  biggest parents:')
for _pid, _r in _top.iterrows():
    print(f'        {str(_names.get(_pid))[:38]:<38} {_pid:<18} {int(_r["n"]):>4} children, '
          f'{int(_r["polys"]):>3} polygons, {"units" if _r["units"] else "children of an area"}')

# the new columns sit with the other derived ones, right after n_osm_bld
_cols = [c for c in pois.columns if c not in ('poi_parent_id', 'split_area_m2')]
_at = _cols.index('n_osm_bld') + 1
pois = pois[_cols[:_at] + ['poi_parent_id', 'split_area_m2'] + _cols[_at:]]
require_unique(pois, 'poi_id', 'pois')
print(f'  ok  POIs with nesting: {len(pois):,} rows, {len(pois.columns)} columns')


  ..  221 area POIs are indoor units (a indoor/level/level:ref key, no building tag) - they can never be parents
      586 point POIs carry a level key too - shop nodes inside malls
  ..  9,285 POIs (34.4 %) sit inside another POI polygon, under 2,528 parents; 1,076 of them under a parent that is itself nested (shop in a mall on a campus)
  ..  1,574 units under 762 building parents; 7,710 children of area parents keep their role; 1 site POIs inside a building polygon stay sites
  ..  poi_role now: {'footprint': 11909, 'point': 11453, 'site': 2063, 'unit': 1574}


  ..  split_area_m2 on all 1,574 units: 189 own area, 73 sibling median, 1,312 fallback 1.0 (even split)
      building parents by child geometry: {'all points': 732, 'all polygons': 17, 'mixed': 13}


  ..  biggest parents:
        nan                                    way/30433298        530 children,  33 polygons, children of an area
        Volkswagenwerk Wolfsburg               way/459317446       234 children, 217 polygons, children of an area
        nan                                    relation/1721133    177 children,  15 polygons, children of an area
        nan                                    way/25489531        167 children,  12 polygons, children of an area
        nan                                    way/47661595        154 children,  17 polygons, children of an area
        Schloss-Arkaden                        way/20018844        128 children, 125 polygons, units
        nan                                    way/196329644       112 children,  91 polygons, children of an area
        nan                                    relation/7314754    105 children,  33 polygons, children of an area
        designer outlets Wolfsburg             way/115793464        93 

## 9. Write both outputs

Both layers are final before either is written. Each file is unlinked first — a
GeoPackage write *appends*, so a stale layer from an earlier run would otherwise
survive next to the new one.

In [10]:
for path, frame, layer in (
    (ALL_POIS_FILE, pois, 'pois'),
    (ALL_BUILDINGS_OSM_FILE, buildings, 'buildings'),
):
    if path.exists():
        try:
            path.unlink()
        except PermissionError as e:
            raise RuntimeError(
                f'{path.name} is locked by another process, so it cannot be '
                'replaced. QGIS holds a GeoPackage open for as long as the '
                'layer is loaded - remove the layer (or close the project) and '
                f'run this cell again. Original error: {e}'
            ) from None
    print(f'Writing {len(frame):,} rows to {path.name} ...', flush=True)
    _t0 = time.perf_counter()
    frame.to_file(path, layer=layer, driver='GPKG')
    print(f'Wrote {len(frame):>9,} rows, {len(frame.columns):>2} cols, '
          f'{path.stat().st_size / 1e6:>6,.1f} MB  -> {path.name} : {layer}'
          f'   [{time.perf_counter() - _t0:.1f}s]')

Writing 26,999 rows to 01_all_pois.gpkg ...


Wrote    26,999 rows, 34 cols,   13.6 MB  -> 01_all_pois.gpkg : pois   [0.5s]
Writing 509,792 rows to 01_all_buildings_osm.gpkg ...


Wrote   509,792 rows, 23 cols,  160.7 MB  -> 01_all_buildings_osm.gpkg : buildings   [6.0s]


## 10. QGIS checkpoint

Load both outputs plus `regionalverband_area.gpkg` and confirm:

1. **Coverage** — both layers blanket the whole region, with no empty
   rectangular holes. A hole means the PBF did not cover the boundary.
2. **Alignment** — buildings sit on a basemap correctly. If everything is in
   the Atlantic near (0, 0), a CRS was mis-assigned rather than reprojected.
3. **Overhang** — the "outside boundary" counts printed above should be small.
4. **Plausibility** — style `pois` by `poi_use`: shops, schools, restaurants and
   surgeries, dense in town centres. If you still see benches and parking, the
   filter in section 5 did not run.
5. **Roles** — filter `poi_role = 'site'`: each should be a campus or complex,
   not a single building. Then filter `poi_role = 'footprint' AND n_osm_bld = 0`
   and check a few against a basemap.
6. **Nesting** — filter `poi_role = 'unit'`: shops inside malls and
   supermarkets, each with a `poi_parent_id`. Filter `poi_parent_id IS NOT NULL
   AND poi_role <> 'unit'`: institutes on campuses, chalets in holiday parks.

`pois` holds points and areas in one layer, so QGIS lists it as two entries,
`pois (Point)` and `pois (Polygon)`. That is a display convention for a generic
GEOMETRY layer, not two layers — load both, or you are looking at a subset.
`buildings` is polygon-only and appears once.

In [11]:
print('STEP 01 SUMMARY')
print('POI funnel:')
print(f'  extracted from PBF    : {n_pois_raw:>8,}')
print(f'  after dropping lines  : {n_pois_geom:>8,}  (-{n_pois_raw - n_pois_geom:,})')
print(f'  after semantic filter : {len(pois):>8,}  (-{n_pois_geom - len(pois):,})')
print(f'  of which from pass 2  : {int(pois["poi_use_tag"].isin(["building", "man_made", "landuse", "railway", "power", "aeroway"]).sum()):>8,}  (named buildings, plants, land use, stations)')
print(f'  kept                  : {100 * len(pois) / n_pois_raw:>7.1f} % of extracted')
print()
print(f'  POIs      : {len(pois):>9,} rows  {len(pois.columns):>2} cols  '
      f'({len(pois_outside):,} outside boundary)')
print(f'  Buildings : {len(buildings):>9,} rows  {len(buildings.columns):>2} cols  '
      f'({len(bld_outside):,} outside boundary)')
print()
_resc = pois['rescued_by'].dropna().map(lambda s: s.split(',')[0].split('=')[0])
print(f'  rescued by evidence or exception: {len(_resc):,}  {_resc.value_counts().to_dict()}')
print()
_nested = pois['poi_parent_id'].notna()
print(f'  nested inside another POI: {int(_nested.sum()):,} under {pois["poi_parent_id"].nunique():,} parents; '
      f'{int((pois["poi_role"] == "unit").sum()):,} of them units of one building (split_area_m2 set)')
print()
print('POI role x geometry:')
print(pd.crosstab(pois['poi_role'], pois['geom_kind']).to_string())
print()
print('poi_use came from:')
print(pois['poi_use_tag'].value_counts(dropna=False).to_string())
print()
print('Top 15 poi_use values:')
print(pois['poi_use'].value_counts().head(15).to_string())
print()
print('Site POIs by real OSM building count:')
site = pois.loc[pois['poi_role'] == 'site', 'n_osm_bld']
if len(site):
    print(pd.cut(site, [1, 2, 5, 20, 100, 10**6],
                 labels=['2', '3-5', '6-20', '21-100', '>100'])
          .value_counts().sort_index().to_string())
print()
print('Buildings with a volume input:')
for c in ('building_levels', 'height'):
    if c in buildings.columns:
        n = int(buildings[c].notna().sum())
        print(f'  {c:<16} {n:>8,}  ({100 * n / len(buildings):5.1f} %)')
print()
print('Load in QGIS:')
for p in (STUDY_BOUNDARY_FILE, ALL_POIS_FILE, ALL_BUILDINGS_OSM_FILE):
    print('  ', p)

STEP 01 SUMMARY
POI funnel:
  extracted from PBF    :   92,779
  after dropping lines  :   92,185  (-594)
  after semantic filter :   26,999  (-65,186)
  of which from pass 2  :    3,904  (named buildings, plants, land use, stations)
  kept                  :    29.1 % of extracted

  POIs      :    26,999 rows  34 cols  (1 outside boundary)
  Buildings :   509,792 rows  23 cols  (0 outside boundary)

  rescued by evidence or exception: 69  {'building': 33, 'recycling_type': 29, 'indoor': 3, 'opening_hours': 2, 'addr:housenumber': 1, 'building:levels': 1}

  nested inside another POI: 9,285 under 2,528 parents; 1,574 of them units of one building (split_area_m2 set)

POI role x geometry:
geom_kind   area  point
poi_role               
footprint  11909      0
point          0  11453
site        2063      0
unit         189   1385

poi_use came from:
poi_use_tag
amenity            9359
shop               6077
tourism            1854
landuse            1698
building           1499
office 